# Modulo 8 · Lezione 10 — Progetto: Classificatore di Immagini con TensorFlow

**Corso di Intelligenza Artificiale — Livello base**

---

## Obiettivo del progetto

Costruire un **classificatore di immagini professionale** per CIFAR-10 (10 categorie: aerei, automobili, uccelli, gatti, cervi, cani, rane, cavalli, navi, camion) applicando tutti i concetti del modulo:

| Fase | Contenuto |
|------|-----------|
| 1 | Esplorazione e preelaborazione del dataset |
| 2 | Baseline: rete densa |
| 3 | CNN da zero con regolarizzazione |
| 4 | Transfer Learning con MobileNetV2 |
| 5 | Confronto e diagnosi finale |
| 6 | Salvare, caricare e usare il modello in produzione |

---

```
CIFAR-10:
  60.000 immagini a colori (RGB) di 32×32 pixel
  10 classi bilanciate, 6.000 immagini per classe
  50.000 train + 10.000 test
  Riferimento: LeCun & Krizhevsky (2009)
```

---
## Fase 1 — Esplorazione e preelaborazione

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import classification_report, confusion_matrix
import os, datetime, json

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {len(gpus)} disponibili {'→ addestramento accelerato' if gpus else '(modalità CPU)'}")

NOMBRES_CLASES = ['aereo','auto','uccello','gatto','cervo','cane','rana','cavallo','nave','camion']

In [ ]:
# Caricare CIFAR-10
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.cifar10.load_data()
y_tr = y_tr_raw.ravel()
y_te = y_te_raw.ravel()

print(f"Train: {X_tr_raw.shape} → {y_tr.shape}")
print(f"Test:  {X_te_raw.shape} → {y_te.shape}")
print(f"Dtype: {X_tr_raw.dtype}, intervallo: [{X_tr_raw.min()}, {X_tr_raw.max()}]")

# Distribuzione delle classi
conteos = np.bincount(y_tr)
print(f"\nClassi (train):")
for i, (nombre, n) in enumerate(zip(NOMBRES_CLASES, conteos)):
    print(f"  {i}: {nombre:10} → {n:,} immagini")

In [ ]:
# Visualizzare campioni del dataset
fig, axes = plt.subplots(10, 8, figsize=(16, 20))
for clase in range(10):
    idx_clase = np.where(y_tr == clase)[0][:8]
    for col, idx in enumerate(idx_clase):
        axes[clase, col].imshow(X_tr_raw[idx])
        if col == 0:
            axes[clase, col].set_ylabel(NOMBRES_CLASES[clase], fontsize=11, rotation=0,
                                         labelpad=55, va='center')
        axes[clase, col].axis('off')
plt.suptitle('CIFAR-10: 8 campioni per classe', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Preelaborazione
# Normalizzare a [0, 1]
X_tr_norm = X_tr_raw.astype('float32') / 255.0
X_te_norm = X_te_raw.astype('float32') / 255.0

# Per i modelli densi: appiattire
X_tr_flat = X_tr_norm.reshape(-1, 32*32*3)
X_te_flat = X_te_norm.reshape(-1, 32*32*3)

# Statistiche
mean = X_tr_norm.mean(axis=(0,1,2))
std  = X_tr_norm.std(axis=(0,1,2))
print(f"Media per canale RGB: {mean.round(4)}")
print(f"Std  per canale RGB:  {std.round(4)}")

# Normalizzazione per canale (per modelli più avanzati)
X_tr_std = (X_tr_norm - mean) / std
X_te_std  = (X_te_norm - mean) / std

print(f"\nDopo la normalizzazione per canale:")
print(f"  Media: {X_tr_std.mean(axis=(0,1,2)).round(4)}")
print(f"  Std:   {X_tr_std.std(axis=(0,1,2)).round(4)}")

---
## Fase 2 — Baseline: rete densa

In [ ]:
# Modello baseline: rete densa semplice
# Scopo: fissare il livello minimo che la CNN deve superare
print("=" * 55)
print("FASE 2: BASELINE — RETE DENSA")
print("=" * 55)

tf.random.set_seed(42)
modelo_dense = keras.Sequential([
    layers.Input(shape=(3072,)),   # 32×32×3
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax'),
], name='baseline_dense')

modelo_dense.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
modelo_dense.summary()

hist_dense = modelo_dense.fit(
    X_tr_flat, y_tr,
    epochs=20, batch_size=256,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

_, acc_dense = modelo_dense.evaluate(X_te_flat, y_te, verbose=0)
print(f"\nBaseline Dense → Test accuracy: {acc_dense:.4f} ({acc_dense:.2%})")

---
## Fase 3 — CNN da zero con regolarizzazione

In [ ]:
print("=" * 55)
print("FASE 3: CNN DA ZERO")
print("=" * 55)

# Data augmentation (si applica solo durante l'addestramento)
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

tf.random.set_seed(42)

entradas = keras.Input(shape=(32, 32, 3))
x = augmentation(entradas)

# Blocco 1: 32×32 → 16×16
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.2)(x)

# Blocco 2: 16×16 → 8×8
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.3)(x)

# Blocco 3: 8×8 → 4×4
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.4)(x)

# Classificatore
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
salidas = layers.Dense(10, activation='softmax')(x)

modelo_cnn = keras.Model(entradas, salidas, name='CNN_cifar10')
modelo_cnn.summary()

In [ ]:
# Compilare con LR schedule
N_EPOCAS   = 60
BATCH_SIZE = 128
pasos_ep   = len(X_tr_norm) * 0.9 // BATCH_SIZE

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001,
    decay_steps=int(N_EPOCAS * pasos_ep),
    alpha=1e-5
)

modelo_cnn.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir_cnn = 'logs/proyecto/cnn/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
callbacks_cnn = [
    keras.callbacks.EarlyStopping('val_accuracy', patience=15, restore_best_weights=True, mode='max'),
    keras.callbacks.ModelCheckpoint('cifar10_cnn_best.keras', monitor='val_accuracy',
                                     save_best_only=True, mode='max', verbose=0),
    keras.callbacks.TensorBoard(log_dir_cnn, histogram_freq=0),
]

hist_cnn = modelo_cnn.fit(
    X_tr_norm, y_tr,
    epochs=N_EPOCAS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks_cnn,
    verbose=1
)

_, acc_cnn = modelo_cnn.evaluate(X_te_norm, y_te, verbose=0)
n_ep_cnn   = len(hist_cnn.history['loss'])
print(f"\nCNN da zero ({n_ep_cnn} epoche) → Test accuracy: {acc_cnn:.4f} ({acc_cnn:.2%})")

In [ ]:
def plot_historial(hist, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist.history['loss'],     'steelblue', lw=2, label='Train')
    axes[0].plot(hist.history['val_loss'], 'coral',     lw=2, label='Val')
    axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{titulo} — Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(hist.history['accuracy'],     'steelblue', lw=2, label='Train')
    axes[1].plot(hist.history['val_accuracy'], 'coral',     lw=2, label='Val')
    axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{titulo} — Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_historial(hist_cnn, 'CNN da zero')

---
## Fase 4 — Transfer Learning con MobileNetV2

In [ ]:
print("=" * 55)
print("FASE 4: TRANSFER LEARNING — MobileNetV2")
print("=" * 55)

# CIFAR-10 è 32×32, MobileNetV2 richiede ≥96×96 → resize
TARGET_SIZE = 96

print(f"Ridimensionamento delle immagini a {TARGET_SIZE}×{TARGET_SIZE}...")
X_tr_up = tf.image.resize(X_tr_norm, (TARGET_SIZE, TARGET_SIZE)).numpy()
X_te_up = tf.image.resize(X_te_norm, (TARGET_SIZE, TARGET_SIZE)).numpy()
print(f"Train: {X_tr_up.shape}")

# Base preaddestrata
base_model = keras.applications.MobileNetV2(
    input_shape=(TARGET_SIZE, TARGET_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
print(f"MobileNetV2 base: {base_model.count_params():,} parametri (congelati)")

# Augmentation per il TL
augm_tl = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Modello TL
tf.random.set_seed(42)
entradas_tl = keras.Input(shape=(TARGET_SIZE, TARGET_SIZE, 3))
x = augm_tl(entradas_tl)
x = keras.applications.mobilenet_v2.preprocess_input(x * 255)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
salidas_tl = layers.Dense(10, activation='softmax')(x)

modelo_tl = keras.Model(entradas_tl, salidas_tl, name='MobileNetV2_TL')

params_trainable = sum(v.numpy().size for v in modelo_tl.trainable_variables)
print(f"Parametri addestrabili: {params_trainable:,} (solo il classificatore)")

# Fase 4a: addestrare solo il classificatore
modelo_tl.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nFase 4a: addestramento del classificatore (base congelata)...")
hist_tl1 = modelo_tl.fit(
    X_tr_up, y_tr,
    epochs=10, batch_size=128,
    validation_split=0.1, verbose=1
)

_, acc_tl1 = modelo_tl.evaluate(X_te_up, y_te, verbose=0)
print(f"\nTL fase 1 → Test accuracy: {acc_tl1:.4f}")

In [ ]:
# Fase 4b: fine-tuning — scongelare gli ultimi strati
print("Fase 4b: fine-tuning (ultimi 30 strati di MobileNetV2)...")

base_model.trainable = True
for capa in base_model.layers[:-30]:
    capa.trainable = False

params_ft = sum(v.numpy().size for v in modelo_tl.trainable_variables)
print(f"Parametri addestrabili dopo il fine-tuning: {params_ft:,}")

# lr molto basso per non distruggere i pesi preaddestrati
modelo_tl.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

hist_tl2 = modelo_tl.fit(
    X_tr_up, y_tr,
    epochs=15, batch_size=64,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping('val_accuracy', patience=8, restore_best_weights=True, mode='max'),
        keras.callbacks.ModelCheckpoint('cifar10_tl_best.keras', monitor='val_accuracy',
                                         save_best_only=True, mode='max', verbose=0),
    ],
    verbose=1
)

_, acc_tl2 = modelo_tl.evaluate(X_te_up, y_te, verbose=0)
print(f"\nTL + fine-tuning → Test accuracy: {acc_tl2:.4f}")

---
## Fase 5 — Confronto e diagnosi finale

In [ ]:
print("=" * 65)
print("FASE 5: CONFRONTO FINALE")
print("=" * 65)

print(f"\n{'Modello':35} {'Params':>12} {'Test acc':>10}")
print('-' * 60)
for nombre, m, acc in [
    ('Baseline Dense',           modelo_dense, acc_dense),
    ('CNN da zero',              modelo_cnn,   acc_cnn),
    ('TL: MobileNetV2 (solo clf)',modelo_tl,   acc_tl1),
    ('TL: MobileNetV2 + FT',     modelo_tl,   acc_tl2),
]:
    p = m.count_params()
    print(f"{nombre:35} {p:>12,} {acc:>10.4f}")

In [ ]:
# Analisi degli errori del modello migliore
mejor_modelo = modelo_tl if acc_tl2 > acc_cnn else modelo_cnn
X_te_mejor   = X_te_up  if acc_tl2 > acc_cnn else X_te_norm
nombre_mejor = 'TL + Fine-tuning' if acc_tl2 > acc_cnn else 'CNN da zero'
acc_mejor    = max(acc_tl2, acc_cnn)

y_pred = mejor_modelo.predict(X_te_mejor, batch_size=256, verbose=0).argmax(axis=1)

# Report
print(f"\nReport del modello migliore — {nombre_mejor}:")
print(classification_report(y_te, y_pred, target_names=NOMBRES_CLASES))

# Matrice di confusione
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=8,
                color='white' if cm[i,j] > cm.max()*0.6 else 'black')
ax.set_xticks(range(10)); ax.set_xticklabels(NOMBRES_CLASES, rotation=45, ha='right')
ax.set_yticks(range(10)); ax.set_yticklabels(NOMBRES_CLASES)
ax.set_xlabel('Predetto'); ax.set_ylabel('Reale')
ax.set_title(f'{nombre_mejor} — Matrice di confusione (acc={acc_mejor:.4f})')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
# Analisi degli errori: quali campioni falliscono?
errores = np.where(y_pred != y_te)[0]
probas  = mejor_modelo.predict(X_te_mejor, batch_size=256, verbose=0)
confianza_errores = probas[errores].max(axis=1)

# Ordinare per alta confidenza (errori più 'sicuri' del modello → peggiori)
orden = np.argsort(-confianza_errores)
peores = errores[orden[:16]]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.ravel(), peores):
    img = X_te_raw[idx]
    ax.imshow(img)
    clase_real  = NOMBRES_CLASES[y_te[idx]]
    clase_pred  = NOMBRES_CLASES[y_pred[idx]]
    conf        = probas[idx].max()
    ax.set_title(f'Reale: {clase_real}\nPred: {clase_pred} ({conf:.2%})',
                 fontsize=8, color='red')
    ax.axis('off')
plt.suptitle('Errori peggiori: alta confidenza in una predizione errata', fontsize=13)
plt.tight_layout(); plt.show()

# Statistiche degli errori
print(f"Errori totali: {len(errores)} / {len(y_te)} ({len(errores)/len(y_te):.2%})")
print(f"Confidenza media negli errori:  {confianza_errores.mean():.4f}")
print(f"Confidenza media nei successi:  {probas[y_pred == y_te].max(axis=1).mean():.4f}")

In [ ]:
# Accuracy per classe + immagini classificate bene e male
print("Accuracy per classe:")
acc_por_clase = []
for c in range(10):
    mask = y_te == c
    acc_c = (y_pred[mask] == y_te[mask]).mean()
    acc_por_clase.append(acc_c)
    print(f"  {NOMBRES_CLASES[c]:10}: {acc_c:.4f} ({acc_c:.2%})")

fig, ax = plt.subplots(figsize=(11, 4))
colores = ['green' if a >= 0.85 else 'orange' if a >= 0.70 else 'red' for a in acc_por_clase]
bars = ax.bar(NOMBRES_CLASES, acc_por_clase, color=colores, edgecolor='white')
ax.axhline(acc_mejor, color='black', ls='--', lw=2, label=f'Accuracy globale ({acc_mejor:.4f})')
ax.set_ylim(0, 1); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy per classe'); ax.legend()
for bar, acc in zip(bars, acc_por_clase):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2%}', ha='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

---
## Fase 6 — Salvare, caricare e usare in produzione

In [ ]:
print("=" * 55)
print("FASE 6: SALVARE E DISTRIBUIRE")
print("=" * 55)

# Salvare il modello migliore
mejor_modelo.save('cifar10_final.keras')
print("Modello salvato: cifar10_final.keras")

# Salvare i metadati del progetto
metadatos = {
    'proyecto':          'CIFAR-10 Classificatore di Immagini',
    'dataset':           'CIFAR-10 (60.000 immagini, 10 classi)',
    'clases':            NOMBRES_CLASES,
    'mejor_modelo':      nombre_mejor,
    'test_accuracy':     float(acc_mejor),
    'baseline_accuracy': float(acc_dense),
    'cnn_accuracy':      float(acc_cnn),
    'tl_accuracy':       float(acc_tl2),
    'input_shape':       [TARGET_SIZE, TARGET_SIZE, 3] if acc_tl2 > acc_cnn else [32, 32, 3],
    'preprocessing':     'normalizzare a [0,1]'
}

with open('cifar10_metadatos.json', 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)
print("Metadati salvati: cifar10_metadatos.json")

In [ ]:
# Funzione di predizione per la produzione
def predecir_imagen(imagen_array, modelo, clases, target_size=None, umbral_confianza=0.7):
    """
    Predice la classe di un'immagine.

    Parametri:
        imagen_array: np.ndarray (H, W, 3) con valori uint8 [0-255]
        modelo:       modello Keras caricato
        clases:       lista dei nomi delle classi
        target_size:  (H, W) se il modello richiede il resize, None per mantenere la dimensione
        umbral_confianza: se la confidenza è inferiore, segnalare "incerto"

    Restituisce:
        dict con classe predetta, confidenza e tutte le probabilità
    """
    # Preelaborare
    img = imagen_array.astype('float32') / 255.0
    if target_size:
        img = tf.image.resize(img, target_size).numpy()

    # Predire
    probas = modelo.predict(img[None, ...], verbose=0)[0]  # (1, H, W, 3) → (n_clases,)
    clase_idx  = probas.argmax()
    confianza  = probas[clase_idx]

    return {
        'clase':     clases[clase_idx],
        'confianza': float(confianza),
        'incierto':  confianza < umbral_confianza,
        'top3': [
            (clases[i], float(probas[i]))
            for i in probas.argsort()[-3:][::-1]
        ]
    }

# Caricare il modello da disco (simuliamo l'uso in produzione)
modelo_cargado = keras.models.load_model('cifar10_final.keras')
print("Modello caricato con successo")

ts = TARGET_SIZE if acc_tl2 > acc_cnn else None

# Provare con 8 immagini
idx_muestra = np.random.choice(len(X_te_raw), 8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), idx_muestra):
    resultado = predecir_imagen(X_te_raw[i], modelo_cargado, NOMBRES_CLASES, target_size=(ts,ts) if ts else None)
    clase_real = NOMBRES_CLASES[y_te[i]]
    ok = resultado['clase'] == clase_real
    ax.imshow(X_te_raw[i])
    titulo = f"Reale: {clase_real}\nPred: {resultado['clase']} ({resultado['confianza']:.0%})"
    ax.set_title(titulo, color='green' if ok else 'red', fontsize=9)
    ax.axis('off')
plt.suptitle('Predizioni del modello caricato da disco', fontsize=13)
plt.tight_layout(); plt.show()

---
## Riepilogo del Progetto

```
CIFAR-10 — Riepilogo dei risultati:

  Baseline Dense:           ~50%  (senza struttura spaziale)
  CNN da zero:               ~80%  (architettura specializzata)
  Transfer Learning (TL):    ~85%  (base di ImageNet)
  TL + Fine-tuning:          ~87%  (ottimizzazione fine degli strati superiori)

Tecniche applicate:
  ✓ Data Augmentation       → più variabilità, meno overfitting
  ✓ BatchNormalization      → addestramento più veloce e stabile
  ✓ Dropout                 → regolarizzazione
  ✓ Regolarizzazione L2     → pesi piccoli
  ✓ GlobalAveragePooling    → meno parametri nel classificatore
  ✓ CosineDecay schedule    → convergenza uniforme
  ✓ EarlyStopping           → fermarsi al momento ottimale
  ✓ ModelCheckpoint         → salvare il modello migliore
  ✓ TensorBoard             → monitoraggio
  ✓ Transfer Learning       → sfruttare ImageNet (14M immagini)
  ✓ Fine-tuning             → adattare gli strati superiori
  ✓ Analisi degli errori    → capire i limiti
```

---
## Sfide aggiuntive

### Sfida 1 — Migliorare il modello

Riesci a superare il 90% su CIFAR-10?

Idee:
- Usare EfficientNetB0 invece di MobileNetV2 (più preciso)
- Aggiungere più strati densi nel classificatore
- Provare ResNet50
- Mixup augmentation: mescolare immagini di due classi
- Label smoothing: `loss='categorical_crossentropy', label_smoothing=0.1`

In [ ]:
# ✏️ IL TUO CODICE QUI


---
### Sfida 2 — Il tuo dataset personale

Applica la stessa pipeline a un problema tuo:

1. Raccogli ≥200 immagini di 2-3 categorie (piante, oggetti, animali...)
2. Organizza in cartelle: `datos/classe1/`, `datos/classe2/`
3. Carica con `keras.utils.image_dataset_from_directory`
4. Applica il Transfer Learning con MobileNetV2
5. Valuta e analizza gli errori

In [ ]:
# ✏️ IL TUO CODICE QUI
# Esempio di caricamento da directory:
# dataset = keras.utils.image_dataset_from_directory(
#     'percorso/ai/tuoi/dati',
#     image_size=(96, 96),
#     batch_size=32,
#     validation_split=0.2,
#     subset='training',
#     seed=42
# )

---
### Sfida 3 — API di predizione

Crea una funzione `predecir_desde_url(url, modelo, clases)` che:

1. Scarichi l'immagine da un URL
2. La preelabori automaticamente
3. Restituisca la predizione con confidenza e top-3
4. Mostri l'immagine con la predizione sovrapposta

Suggerimento: usa `requests`, `PIL.Image`, `io.BytesIO`

In [ ]:
# ✏️ IL TUO CODICE QUI


---
## Riepilogo del Modulo 8 — Deep Learning

```
L1  Storia e fondamenti       → percettrone, reti multistrato, storia del DL
L2  Neuroni artificiali       → attivazioni, inizializzazione, forward pass
L3  Backpropagation           → gradiente, ottimizzatori, Adam
L4  TensorFlow e Keras        → Sequential, compile, fit, evaluate
L5  Reti dense                → classificazione binaria/multiclasse, diagnosi
L6  Regolarizzazione          → Dropout, BatchNorm, L2, EarlyStopping
L7  CNN                       → Conv2D, Pooling, Transfer Learning
L8  RNN e Transformer         → LSTM, GRU, Embedding, attenzione
L9  Addestramento             → TensorBoard, LR schedule, callback
L10 Progetto                  → CIFAR-10 end-to-end: Dense→CNN→TL→FT
```

### Cosa viene nel Modulo 9?

Nel prossimo modulo impareremo a **distribuire i modelli in produzione**: API con FastAPI, container Docker, monitoraggio in produzione e le basi del MLOps.